# Phase 5: OCR & Document Intelligence
## Day 24: OcrLlmPipeline

Date: 2026-04-24

### Learning objectives
- Build an image to JSON document pipeline.
- Preprocess noisy document images.
- Run OCR with safe fallbacks.
- Use an LLM-style extractor to structure OCR text.
- Validate, repair, and score extracted fields.
- Handle noisy OCR text with cleanup and retry patterns.

In [ ]:
import json
import re
import shutil
import textwrap
from pprint import pprint
from typing import Optional, Dict, Any, List

import numpy as np
import pandas as pd

try:
    import cv2
    CV2_AVAILABLE = True
except Exception:
    cv2 = None
    CV2_AVAILABLE = False

try:
    from PIL import Image, ImageDraw, ImageFont, ImageFilter
    PIL_AVAILABLE = True
except Exception:
    PIL_AVAILABLE = False

try:
    import pytesseract
    PYTESSERACT_AVAILABLE = True
except Exception:
    pytesseract = None
    PYTESSERACT_AVAILABLE = False

try:
    from pydantic import BaseModel, Field, ValidationError
    PYDANTIC_AVAILABLE = True
except Exception:
    BaseModel = object
    Field = None
    ValidationError = Exception
    PYDANTIC_AVAILABLE = False

TESSERACT_BINARY_AVAILABLE = shutil.which("tesseract") is not None

def show(title, content):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    print(textwrap.dedent(str(content)).strip())

def model_to_dict(model):
    if hasattr(model, "model_dump"):
        return model.model_dump()
    if hasattr(model, "dict"):
        return model.dict()
    return model

print("Setup complete.")
print("OpenCV available:", CV2_AVAILABLE)
print("PIL available:", PIL_AVAILABLE)
print("pytesseract available:", PYTESSERACT_AVAILABLE)
print("Tesseract binary available:", TESSERACT_BINARY_AVAILABLE)
print("Pydantic available:", PYDANTIC_AVAILABLE)

In [ ]:
invoice_documents = [
    {
        "doc_id": "DOC-001",
        "kind": "clean",
        "text": '''
        INVOICE
        Invoice ID: INV-5001
        Vendor: Berlin Coffee Bar
        Date: 2026-04-24
        Campaign: Spring Coffee Push
        Channel: Instagram
        Spend: 1200 EUR
        Clicks: 3420
        Conversions: 184
        Total: 1200 EUR
        '''
    },
    {
        "doc_id": "DOC-002",
        "kind": "noisy",
        "text": '''
        INVOICE
        Invoice ID: INV-5002
        Vendor: Yoga Studio Berlin
        Date: 2026-04-23
        Campaign: Yoga Studio Trial
        Channel: TikTok
        Spend: 650 EUR
        Clicks: 2100
        Conversions: 165
        Total: 650 EUR
        '''
    },
    {
        "doc_id": "DOC-003",
        "kind": "skewed",
        "text": '''
        INVOICE
        Invoice ID: INV-5003
        Vendor: Bank App Team
        Date: 2026-04-22
        Campaign: Bank App Onboarding
        Channel: Email
        Spend: 800 EUR
        Clicks: 980
        Conversions: 42
        Total: 800 EUR
        '''
    }
]

target_schema = {
    "doc_id": "string",
    "invoice_id": "string",
    "vendor": "string or null",
    "date": "YYYY-MM-DD string or null",
    "campaign": "string or null",
    "channel": "string or null",
    "spend_eur": "number or null",
    "clicks": "integer or null",
    "conversions": "integer or null",
    "total_eur": "number or null"
}

show("First sample document text", invoice_documents[0]["text"])
print("\nTarget schema:")
print(json.dumps(target_schema, indent=2))

## 1. Pipeline overview

The full document intelligence pipeline has five steps.

Image, preprocessing, OCR, LLM extraction, then validation.

In [ ]:
pipeline_steps = pd.DataFrame([
    {"step": 1, "name": "Image input", "goal": "Load or create a document image"},
    {"step": 2, "name": "Preprocess", "goal": "Improve contrast, remove noise, deskew"},
    {"step": 3, "name": "OCR", "goal": "Convert image text into raw text"},
    {"step": 4, "name": "LLM extraction", "goal": "Convert raw OCR text into JSON"},
    {"step": 5, "name": "Validation", "goal": "Check fields, types, and business rules"},
])

pipeline_steps

## 2. Create document images

We generate document images from text.

Then we make clean, noisy, and skewed versions to simulate real OCR problems.

In [ ]:
def create_document_image(text, width=900, height=500, font_size=24):
    if not PIL_AVAILABLE:
        return None

    image = Image.new("RGB", (width, height), color="white")
    draw = ImageDraw.Draw(image)

    try:
        font = ImageFont.truetype("DejaVuSansMono.ttf", font_size)
    except Exception:
        font = ImageFont.load_default()

    draw.multiline_text((40, 35), text.strip(), fill="black", font=font, spacing=10)
    return image

def pil_to_rgb_array(image):
    if image is None:
        return None
    return np.array(image.convert("RGB"))

def rgb_array_to_pil(array):
    if array is None or not PIL_AVAILABLE:
        return None
    return Image.fromarray(np.clip(array, 0, 255).astype(np.uint8))

def display_image(array_or_pil):
    if array_or_pil is None:
        print("No image to display.")
        return

    if PIL_AVAILABLE:
        if isinstance(array_or_pil, Image.Image):
            display(array_or_pil)
        else:
            if len(array_or_pil.shape) == 2:
                display(Image.fromarray(np.clip(array_or_pil, 0, 255).astype(np.uint8)))
            else:
                display(rgb_array_to_pil(array_or_pil))
    else:
        print("PIL is not available.")

clean_image_pil = create_document_image(invoice_documents[0]["text"])
clean_rgb = pil_to_rgb_array(clean_image_pil)

display_image(clean_image_pil)

In [ ]:
def add_noise(image_rgb, noise_std=28, seed=42):
    if image_rgb is None:
        return None

    rng = np.random.default_rng(seed)
    noise = rng.normal(0, noise_std, image_rgb.shape)
    noisy = image_rgb.astype(np.float32) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

def rotate_image(image_rgb, angle_degrees):
    if image_rgb is None:
        return None

    if CV2_AVAILABLE:
        height, width = image_rgb.shape[:2]
        center = (width // 2, height // 2)
        matrix = cv2.getRotationMatrix2D(center, angle_degrees, 1.0)
        return cv2.warpAffine(image_rgb, matrix, (width, height), borderValue=(255, 255, 255))

    if PIL_AVAILABLE:
        pil = rgb_array_to_pil(image_rgb)
        return np.array(pil.rotate(angle_degrees, expand=False, fillcolor="white"))

    return image_rgb

document_images = {}

for doc in invoice_documents:
    base_pil = create_document_image(doc["text"])
    base_rgb = pil_to_rgb_array(base_pil)

    if doc["kind"] == "noisy":
        image_rgb = add_noise(base_rgb, noise_std=34, seed=7)
    elif doc["kind"] == "skewed":
        image_rgb = rotate_image(base_rgb, angle_degrees=6)
    else:
        image_rgb = base_rgb

    document_images[doc["doc_id"]] = image_rgb

print("Created images:", list(document_images.keys()))
display_image(document_images["DOC-002"])

## 3. Preprocess the image

Preprocessing makes text easier to read.

Today we combine grayscale, denoising, thresholding, and optional deskewing.

In [ ]:
def to_grayscale(image_rgb):
    if image_rgb is None:
        return None

    if CV2_AVAILABLE:
        return cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

    return np.dot(image_rgb[..., :3], [0.299, 0.587, 0.114]).astype(np.uint8)

def denoise_image(gray_image):
    if gray_image is None:
        return None

    if CV2_AVAILABLE:
        return cv2.fastNlMeansDenoising(gray_image, None, h=18, templateWindowSize=7, searchWindowSize=21)

    if PIL_AVAILABLE:
        pil = Image.fromarray(gray_image.astype(np.uint8))
        return np.array(pil.filter(ImageFilter.MedianFilter(size=3)))

    return gray_image

def otsu_threshold(gray_image):
    if gray_image is None:
        return None, None

    if CV2_AVAILABLE:
        value, binary = cv2.threshold(gray_image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return binary, value

    threshold = int(np.mean(gray_image))
    binary = np.where(gray_image > threshold, 255, 0).astype(np.uint8)
    return binary, threshold

def adaptive_threshold(gray_image, block_size=35, c_value=11):
    if gray_image is None:
        return None

    if block_size % 2 == 0:
        block_size += 1

    if CV2_AVAILABLE:
        return cv2.adaptiveThreshold(
            gray_image,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            block_size,
            c_value
        )

    return np.where(gray_image > np.mean(gray_image), 255, 0).astype(np.uint8)

gray_clean = to_grayscale(document_images["DOC-001"])
denoised_clean = denoise_image(gray_clean)
binary_clean, threshold_value = otsu_threshold(denoised_clean)

print("Otsu threshold value:", threshold_value)
display_image(binary_clean)

In [ ]:
def estimate_skew_angle(gray_image):
    if gray_image is None or not CV2_AVAILABLE:
        return 0.0

    _, binary = cv2.threshold(gray_image, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = np.column_stack(np.where(binary > 0))

    if len(coords) < 10:
        return 0.0

    angle = cv2.minAreaRect(coords)[-1]

    if angle < -45:
        angle = 90 + angle

    return float(angle)

def deskew_rgb(image_rgb):
    if image_rgb is None:
        return None, 0.0

    gray_image = to_grayscale(image_rgb)
    angle = estimate_skew_angle(gray_image)
    corrected = rotate_image(image_rgb, angle_degrees=angle)
    return corrected, angle

skewed_rgb = document_images["DOC-003"]
deskewed_rgb, detected_angle = deskew_rgb(skewed_rgb)

print("Detected correction angle:", round(detected_angle, 2))
display_image(deskewed_rgb)

In [ ]:
def preprocess_document_image(image_rgb, method="otsu", denoise=True, deskew=False):
    if image_rgb is None:
        return {
            "preprocessed_image": None,
            "deskew_angle": 0.0,
            "method": method,
            "threshold_value": None
        }

    working_rgb = image_rgb

    if deskew:
        working_rgb, angle = deskew_rgb(working_rgb)
    else:
        angle = 0.0

    gray = to_grayscale(working_rgb)

    if denoise:
        gray = denoise_image(gray)

    if method == "adaptive":
        processed = adaptive_threshold(gray)
        threshold_value = "adaptive"
    elif method == "otsu":
        processed, threshold_value = otsu_threshold(gray)
    else:
        processed = gray
        threshold_value = None

    return {
        "preprocessed_image": processed,
        "deskew_angle": angle,
        "method": method,
        "threshold_value": threshold_value
    }

preprocessed_examples = {
    doc_id: preprocess_document_image(
        image_rgb,
        method="otsu",
        denoise=True,
        deskew=(doc_id == "DOC-003")
    )
    for doc_id, image_rgb in document_images.items()
}

for doc_id, result in preprocessed_examples.items():
    print(doc_id, {k: v for k, v in result.items() if k != "preprocessed_image"})

## 4. OCR step with fallback

Real OCR needs Tesseract installed.

This notebook uses real OCR when available. Otherwise, it uses the original text as a mock OCR output.

In [ ]:
def find_original_text(doc_id):
    for doc in invoice_documents:
        if doc["doc_id"] == doc_id:
            return doc["text"].strip()
    return ""

def simulate_ocr_noise(text):
    replacements = {
        "Invoice ID": "lnvoice ID",
        "Conversions": "Convers1ons",
        "Total": "TotaI",
        "Spend": "5pend",
        "Clicks": "C1icks",
    }
    noisy = text
    for old, new in replacements.items():
        noisy = noisy.replace(old, new)
    return noisy

def run_ocr(preprocessed_image, doc_id, noisy=False, lang="eng", config="--psm 6 --oem 3"):
    if preprocessed_image is not None and PYTESSERACT_AVAILABLE and TESSERACT_BINARY_AVAILABLE:
        try:
            pil_image = Image.fromarray(preprocessed_image.astype(np.uint8)) if PIL_AVAILABLE else preprocessed_image
            return pytesseract.image_to_string(pil_image, lang=lang, config=config)
        except Exception as error:
            print("Real OCR failed. Using mock OCR.")
            print("Error:", error)

    text = find_original_text(doc_id)
    return simulate_ocr_noise(text) if noisy else text

ocr_outputs = {}

for doc in invoice_documents:
    doc_id = doc["doc_id"]
    noisy = doc_id == "DOC-002"
    ocr_text = run_ocr(
        preprocessed_examples[doc_id]["preprocessed_image"],
        doc_id=doc_id,
        noisy=noisy
    )
    ocr_outputs[doc_id] = ocr_text

show("OCR output for noisy document", ocr_outputs["DOC-002"])

## 5. Clean noisy OCR text

OCR text often contains small character mistakes.

A cleanup layer fixes common OCR confusions before extraction.

In [ ]:
def clean_ocr_text(text):
    text = text.replace("\x0c", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()

def repair_common_ocr_errors(text):
    replacements = {
        "lnvoice ID": "Invoice ID",
        "Convers1ons": "Conversions",
        "TotaI": "Total",
        "5pend": "Spend",
        "C1icks": "Clicks",
        "EUR0": "EUR",
    }
    repaired = text
    for wrong, correct in replacements.items():
        repaired = repaired.replace(wrong, correct)
    return repaired

raw_noisy = ocr_outputs["DOC-002"]
cleaned_noisy = clean_ocr_text(raw_noisy)
repaired_noisy = repair_common_ocr_errors(cleaned_noisy)

show("Raw noisy OCR", raw_noisy)
show("Repaired noisy OCR", repaired_noisy)

In [ ]:
def ocr_text_quality_score(text):
    required_keywords = ["Invoice ID", "Vendor", "Date", "Campaign", "Channel", "Spend", "Clicks", "Conversions", "Total"]
    clean = repair_common_ocr_errors(clean_ocr_text(text))
    found = sum(keyword in clean for keyword in required_keywords)

    return {
        "found_keywords": found,
        "total_keywords": len(required_keywords),
        "score": round(found / len(required_keywords), 3),
        "missing": [keyword for keyword in required_keywords if keyword not in clean]
    }

for doc_id, text in ocr_outputs.items():
    print("\n", doc_id)
    pprint(ocr_text_quality_score(text))

## 6. Build the LLM extraction prompt

The LLM should receive the OCR text and a strict JSON schema.

The prompt asks for JSON only because the next step is parsing.

In [ ]:
def build_llm_extraction_prompt(doc_id, ocr_text, schema):
    return f'''
Task:
Extract invoice and campaign fields from OCR text.

Document ID:
{doc_id}

OCR text:
{repair_common_ocr_errors(clean_ocr_text(ocr_text))}

Rules:
- Return valid JSON only.
- Do not include markdown.
- Do not include explanations.
- Use null for missing values.
- Numeric fields must be numbers.
- Date must use YYYY-MM-DD when available.

JSON schema:
{json.dumps(schema, indent=2)}
'''.strip()

prompt = build_llm_extraction_prompt("DOC-002", ocr_outputs["DOC-002"], target_schema)
show("LLM extraction prompt", prompt)

## 7. Mock LLM extraction

In production, this step calls an LLM.

Here we use a regex-based mock LLM so every cell runs locally.

In [ ]:
def extract_field(pattern, text, flags=re.IGNORECASE):
    match = re.search(pattern, text, flags=flags)
    return match.group(1).strip() if match else None

def extract_number(pattern, text):
    value = extract_field(pattern, text)
    return int(value) if value is not None else None

def mock_llm_extract_json(doc_id, ocr_text):
    text = repair_common_ocr_errors(clean_ocr_text(ocr_text))

    record = {
        "doc_id": doc_id,
        "invoice_id": extract_field(r"Invoice ID:\s*([A-Z]+-[0-9]+)", text),
        "vendor": extract_field(r"Vendor:\s*(.+)", text),
        "date": extract_field(r"Date:\s*(\d{4}-\d{2}-\d{2})", text),
        "campaign": extract_field(r"Campaign:\s*(.+)", text),
        "channel": extract_field(r"Channel:\s*([A-Za-z]+)", text),
        "spend_eur": extract_number(r"Spend:\s*(\d+)\s*EUR", text),
        "clicks": extract_number(r"Clicks:\s*(\d+)", text),
        "conversions": extract_number(r"Conversions:\s*(\d+)", text),
        "total_eur": extract_number(r"Total:\s*(\d+)\s*EUR", text),
    }

    return json.dumps(record, indent=2)

mock_output = mock_llm_extract_json("DOC-002", ocr_outputs["DOC-002"])
print(mock_output)

## 8. Parse, repair, and validate JSON

LLM output can be broken.

We parse safely, repair small formatting issues, and validate all important fields.

In [ ]:
def safe_json_loads(text):
    try:
        return json.loads(text), None
    except json.JSONDecodeError as error:
        return None, str(error)

def extract_json_block(text):
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    return match.group(0) if match else None

def repair_json_text(text):
    repaired = text.strip()
    repaired = repaired.replace("```json", "").replace("```", "").strip()
    repaired = re.sub(r",\s*}", "}", repaired)
    repaired = re.sub(r",\s*]", "]", repaired)
    return repaired

broken_json = '''
```json
{
  "doc_id": "DOC-002",
  "invoice_id": "INV-5002",
  "vendor": "Yoga Studio Berlin",
  "date": "2026-04-23",
  "campaign": "Yoga Studio Trial",
  "channel": "TikTok",
  "spend_eur": 650,
  "clicks": 2100,
  "conversions": 165,
  "total_eur": 650,
}
```
'''

json_block = extract_json_block(broken_json)
repaired_json = repair_json_text(json_block)
data, error = safe_json_loads(repaired_json)

pprint(data)
print("Error:", error)

In [ ]:
if PYDANTIC_AVAILABLE:
    class InvoiceCampaignRecord(BaseModel):
        doc_id: str
        invoice_id: Optional[str] = None
        vendor: Optional[str] = None
        date: Optional[str] = None
        campaign: Optional[str] = None
        channel: Optional[str] = None
        spend_eur: Optional[float] = Field(default=None, ge=0)
        clicks: Optional[int] = Field(default=None, ge=0)
        conversions: Optional[int] = Field(default=None, ge=0)
        total_eur: Optional[float] = Field(default=None, ge=0)
else:
    InvoiceCampaignRecord = None

def fallback_validate_record(data):
    required = [
        "doc_id", "invoice_id", "vendor", "date", "campaign", "channel",
        "spend_eur", "clicks", "conversions", "total_eur"
    ]
    errors = []

    for field in required:
        if field not in data:
            errors.append(f"Missing field: {field}")

    for field in ["spend_eur", "clicks", "conversions", "total_eur"]:
        value = data.get(field)
        if value is not None and not isinstance(value, (int, float)):
            errors.append(f"{field} must be numeric or null")
        if isinstance(value, (int, float)) and value < 0:
            errors.append(f"{field} must be non-negative")

    if data.get("clicks") is not None and data.get("conversions") is not None:
        if data["conversions"] > data["clicks"]:
            errors.append("conversions cannot be greater than clicks")

    if data.get("date") is not None and not re.match(r"^\d{4}-\d{2}-\d{2}$", data["date"]):
        errors.append("date must use YYYY-MM-DD")

    if errors:
        raise ValueError(errors)

    return data

def validate_record(data):
    if PYDANTIC_AVAILABLE:
        model = InvoiceCampaignRecord(**data)
        model_data = model_to_dict(model)
        fallback_validate_record(model_data)
        return model
    return fallback_validate_record(data)

validated = validate_record(data)
pprint(model_to_dict(validated))

## 9. Retry pattern for noisy text

Sometimes cleanup is not enough.

A retry prompt gives the model the original OCR text, the bad output, and the validation error.

In [ ]:
def make_retry_prompt(doc_id, ocr_text, bad_output, error_message, schema):
    return f'''
The previous extraction failed.

Document ID:
{doc_id}

OCR text:
{repair_common_ocr_errors(clean_ocr_text(ocr_text))}

Bad output:
{bad_output}

Error:
{error_message}

Return a corrected JSON object.

Rules:
- Return valid JSON only.
- Do not include markdown.
- Do not include explanations.
- Use this exact schema:
{json.dumps(schema, indent=2)}
'''.strip()

bad_output = '{"doc_id": "DOC-002", "invoice_id": "INV-5002", "clicks": "many",}'
_, parse_error = safe_json_loads(bad_output)

retry_prompt = make_retry_prompt(
    "DOC-002",
    ocr_outputs["DOC-002"],
    bad_output,
    parse_error,
    target_schema
)

show("Retry prompt", retry_prompt)

In [ ]:
def parse_validate_or_retry(doc_id, ocr_text, llm_output):
    json_text = extract_json_block(llm_output) or llm_output
    json_text = repair_json_text(json_text)

    data, parse_error = safe_json_loads(json_text)

    if parse_error:
        return None, f"Parse error: {parse_error}", True

    try:
        validated = validate_record(data)
        return model_to_dict(validated), None, False
    except Exception as validation_error:
        return None, f"Validation error: {validation_error}", True

record, error, needs_retry = parse_validate_or_retry("DOC-002", ocr_outputs["DOC-002"], mock_output)

print("Needs retry:", needs_retry)
print("Error:", error)
pprint(record)

## 10. Build the full OCR plus LLM pipeline

Now we connect every step.

The output contains the extracted record, errors, quality score, and pipeline metadata.

In [ ]:
def run_ocr_llm_pipeline(doc, image_rgb, method="otsu", denoise=True, deskew=False, simulate_broken_json=False):
    doc_id = doc["doc_id"]

    preprocess_result = preprocess_document_image(
        image_rgb,
        method=method,
        denoise=denoise,
        deskew=deskew
    )

    ocr_text = run_ocr(
        preprocess_result["preprocessed_image"],
        doc_id=doc_id,
        noisy=(doc["kind"] == "noisy")
    )

    quality = ocr_text_quality_score(ocr_text)
    prompt = build_llm_extraction_prompt(doc_id, ocr_text, target_schema)
    llm_output = mock_llm_extract_json(doc_id, ocr_text)

    if simulate_broken_json:
        llm_output = llm_output[:-2] + ",\n}"

    record, error, needs_retry = parse_validate_or_retry(doc_id, ocr_text, llm_output)
    retry_used = False

    if needs_retry:
        retry_used = True
        retry_output = mock_llm_extract_json(doc_id, repair_common_ocr_errors(ocr_text))
        record, error, _ = parse_validate_or_retry(doc_id, ocr_text, retry_output)
    else:
        retry_output = None

    return {
        "doc_id": doc_id,
        "preprocess": {
            "method": preprocess_result["method"],
            "deskew_angle": preprocess_result["deskew_angle"],
            "threshold_value": preprocess_result["threshold_value"]
        },
        "ocr_text": ocr_text,
        "quality": quality,
        "prompt_preview": prompt[:200] + "...",
        "llm_output": llm_output,
        "retry_used": retry_used,
        "retry_output": retry_output,
        "record": record,
        "error": error
    }

pipeline_result = run_ocr_llm_pipeline(
    invoice_documents[1],
    document_images["DOC-002"],
    method="otsu",
    denoise=True,
    deskew=False
)

pprint({k: v for k, v in pipeline_result.items() if k not in ["ocr_text", "llm_output", "retry_output"]})

In [ ]:
all_results = []

for doc in invoice_documents:
    result = run_ocr_llm_pipeline(
        doc,
        document_images[doc["doc_id"]],
        method="otsu",
        denoise=True,
        deskew=(doc["kind"] == "skewed"),
        simulate_broken_json=(doc["doc_id"] == "DOC-003")
    )
    all_results.append(result)

for result in all_results:
    print("\nDocument:", result["doc_id"])
    print("Quality score:", result["quality"]["score"])
    print("Retry used:", result["retry_used"])
    print("Error:", result["error"])
    pprint(result["record"])

## 11. Analyze the structured output

Once extraction is complete, the result becomes normal tabular data.

This is the main value of document intelligence.

In [ ]:
records = [result["record"] for result in all_results if result["error"] is None]
df = pd.DataFrame(records)

df["conversion_rate"] = df["conversions"] / df["clicks"]
df["spend_matches_total"] = abs(df["spend_eur"] - df["total_eur"]) < 0.01

df

In [ ]:
summary = {
    "documents_processed": len(all_results),
    "successful_records": int(sum(result["error"] is None for result in all_results)),
    "retry_count": int(sum(result["retry_used"] for result in all_results)),
    "avg_ocr_quality": round(float(np.mean([result["quality"]["score"] for result in all_results])), 3),
    "total_spend": round(float(df["spend_eur"].sum()), 2),
    "total_conversions": int(df["conversions"].sum())
}

pprint(summary)

## 12. Quality checks

Use simple rules to catch extraction mistakes.

These checks are cheap and useful before storing results.

In [ ]:
def run_quality_checks(df):
    checks = {
        "invoice_id_present": df["invoice_id"].notna().all(),
        "date_present": df["date"].notna().all(),
        "spend_non_negative": (df["spend_eur"] >= 0).all(),
        "clicks_non_negative": (df["clicks"] >= 0).all(),
        "conversions_not_above_clicks": (df["conversions"] <= df["clicks"]).all(),
        "spend_matches_total": df["spend_matches_total"].all(),
    }
    return checks

checks = run_quality_checks(df)
pprint(checks)

assert all(checks.values())
print("All quality checks passed.")

## Tricky bits

OCR plus LLM pipelines can fail in several places.

Good pipelines keep each step separate so you know where the problem happened.

In [ ]:
failure_modes = pd.DataFrame([
    {
        "stage": "Image",
        "problem": "Low resolution or blur",
        "fix": "Improve scan quality or upscale carefully"
    },
    {
        "stage": "Preprocessing",
        "problem": "Threshold removes light text",
        "fix": "Try adaptive thresholding or less aggressive preprocessing"
    },
    {
        "stage": "OCR",
        "problem": "Characters are confused",
        "fix": "Repair common OCR errors and use right language pack"
    },
    {
        "stage": "LLM extraction",
        "problem": "JSON has prose or missing fields",
        "fix": "Use strict schema and retry prompt"
    },
    {
        "stage": "Validation",
        "problem": "Types or business rules fail",
        "fix": "Reject, retry, or send to manual review"
    },
])

failure_modes

In [ ]:
def diagnose_pipeline_result(result):
    if result["quality"]["score"] < 0.7:
        return "OCR quality is low. Improve preprocessing or image quality."
    if result["error"] is not None:
        return "Extraction or validation failed. Check retry prompt and schema."
    if result["retry_used"]:
        return "Pipeline succeeded after retry. Review why first output failed."
    return "Pipeline looks healthy."

for result in all_results:
    print(result["doc_id"], "=>", diagnose_pipeline_result(result))

## Trick questions

1. Why split OCR and LLM extraction into separate steps?

<details>
<summary>Answer</summary>

It makes debugging easier. You can see whether the problem came from image quality, OCR text, extraction, or validation.

</details>

2. Should the LLM receive the image or OCR text in this pipeline?

<details>
<summary>Answer</summary>

In this notebook, the LLM receives OCR text. Some modern vision models can read images directly, but OCR text is easier to validate and debug.

</details>

3. Why repair OCR text before extraction?

<details>
<summary>Answer</summary>

Small OCR errors can break field extraction. Fixing common errors improves the JSON output.

</details>

4. Why validate after JSON parsing?

<details>
<summary>Answer</summary>

Parsing only checks JSON syntax. Validation checks fields, types, dates, numbers, and business rules.

</details>

5. What should happen if validation fails after retry?

<details>
<summary>Answer</summary>

Mark the document as failed or send it to manual review. Do not silently trust bad data.

</details>

## Exercises

Fill in each `___`. Run the cell to check your answer.

In [ ]:
# Exercise 1
# Preprocess the clean document image with Otsu thresholding.

prep = ___

assert isinstance(prep, dict)
assert "preprocessed_image" in prep
assert prep["method"] == "otsu"
print("Exercise 1 passed.")

In [ ]:
# Exercise 2
# Run OCR on DOC-001 using the helper.

ocr_text = ___

assert isinstance(ocr_text, str)
assert "Invoice ID" in ocr_text
print("Exercise 2 passed.")

In [ ]:
# Exercise 3
# Repair common OCR errors.

bad_text = "lnvoice ID: INV-1\n5pend: 100 EUR\nC1icks: 200\nConvers1ons: 20\nTotaI: 100 EUR"
fixed_text = ___

assert "Invoice ID" in fixed_text
assert "Spend" in fixed_text
assert "Clicks" in fixed_text
assert "Conversions" in fixed_text
assert "Total" in fixed_text
print("Exercise 3 passed.")

In [ ]:
# Exercise 4
# Build an LLM extraction prompt.

prompt = ___

assert isinstance(prompt, str)
assert "Return valid JSON only" in prompt
assert "DOC-001" in prompt
print("Exercise 4 passed.")

In [ ]:
# Exercise 5
# Run mock LLM extraction and parse the JSON.

llm_output = mock_llm_extract_json("DOC-001", ocr_outputs["DOC-001"])
data, error = ___

assert error is None
assert data["invoice_id"] == "INV-5001"
print("Exercise 5 passed.")

In [ ]:
# Exercise 6
# Validate the parsed data.

validated = ___

assert model_to_dict(validated)["doc_id"] == "DOC-001"
assert model_to_dict(validated)["spend_eur"] == 1200
print("Exercise 6 passed.")

In [ ]:
# Exercise 7
# Run the full OCR plus LLM pipeline for DOC-003.

result = ___

assert result["doc_id"] == "DOC-003"
assert result["error"] is None
assert result["record"]["invoice_id"] == "INV-5003"
print("Exercise 7 passed.")

## Solutions

<details>
<summary>Exercise 1 solution</summary>

```python
prep = preprocess_document_image(
    document_images["DOC-001"],
    method="otsu",
    denoise=True,
    deskew=False
)
```

</details>

<details>
<summary>Exercise 2 solution</summary>

```python
ocr_text = run_ocr(
    prep["preprocessed_image"],
    doc_id="DOC-001",
    noisy=False
)
```

</details>

<details>
<summary>Exercise 3 solution</summary>

```python
fixed_text = repair_common_ocr_errors(bad_text)
```

</details>

<details>
<summary>Exercise 4 solution</summary>

```python
prompt = build_llm_extraction_prompt(
    "DOC-001",
    ocr_outputs["DOC-001"],
    target_schema
)
```

</details>

<details>
<summary>Exercise 5 solution</summary>

```python
data, error = safe_json_loads(llm_output)
```

</details>

<details>
<summary>Exercise 6 solution</summary>

```python
validated = validate_record(data)
```

</details>

<details>
<summary>Exercise 7 solution</summary>

```python
result = run_ocr_llm_pipeline(
    invoice_documents[2],
    document_images["DOC-003"],
    method="otsu",
    denoise=True,
    deskew=True
)
```

</details>

## Cumulative review exercises

These mix topics from Days 14 to 23. Fill in `___` and run each cell.

In [ ]:
# Review 1: Fine-tuning BERT
# Pick the common Hugging Face class used to train models.

trainer_class = ___

assert trainer_class == "Trainer"
print("Review 1 passed.")

In [ ]:
# Review 2: Complaint classification
# Create a label mapping.

label_to_id = ___

assert isinstance(label_to_id, dict)
assert len(label_to_id) >= 3
assert all(isinstance(value, int) for value in label_to_id.values())
print("Review 2 passed.")

In [ ]:
# Review 3: OpenAI API
# Fill the standard chat roles.

roles = ___

assert roles == ["system", "user", "assistant"]
print("Review 3 passed.")

In [ ]:
# Review 4: Ollama
# Fill the default local generate endpoint.

ollama_url = ___

assert ollama_url == "http://localhost:11434/api/generate"
print("Review 4 passed.")

In [ ]:
# Review 5: Prompt engineering
# Choose the prompting style that uses examples.

prompt_style = ___

assert prompt_style.lower() == "few-shot"
print("Review 5 passed.")

In [ ]:
# Review 6: Structured output
# Parse JSON text.

json_text = '{"campaign": "Demo", "clicks": 100}'
parsed = ___

assert parsed["clicks"] == 100
print("Review 6 passed.")

In [ ]:
# Review 7: Information extraction
# Calculate conversion rate.

record = {"clicks": 2000, "conversions": 160}
conversion_rate = ___

assert abs(conversion_rate - 0.08) < 1e-9
print("Review 7 passed.")

In [ ]:
# Review 8: Tesseract basics
# Choose the Tesseract language code for German.

german_lang_code = ___

assert german_lang_code == "deu"
print("Review 8 passed.")

In [ ]:
# Review 9: EasyOCR
# Create an EasyOCR language list for English and Turkish.

easyocr_languages = ___

assert easyocr_languages == ["en", "tr"] or easyocr_languages == ["tr", "en"]
print("Review 9 passed.")

In [ ]:
# Review 10: OpenCV preprocessing
# Convert image to grayscale.

review_gray = ___

assert review_gray is not None
assert len(review_gray.shape) == 2
print("Review 10 passed.")

## Cumulative review solutions

<details>
<summary>Show solutions</summary>

```python
# Review 1
trainer_class = "Trainer"

# Review 2
label_to_id = {"billing": 0, "delivery": 1, "technical": 2}

# Review 3
roles = ["system", "user", "assistant"]

# Review 4
ollama_url = "http://localhost:11434/api/generate"

# Review 5
prompt_style = "few-shot"

# Review 6
parsed = json.loads(json_text)

# Review 7
conversion_rate = record["conversions"] / record["clicks"]

# Review 8
german_lang_code = "deu"

# Review 9
easyocr_languages = ["en", "tr"]

# Review 10
review_gray = to_grayscale(document_images["DOC-001"])
```

</details>

In [ ]:
cheat_sheet = '''
DAY 24 CHEAT SHEET: OCR PLUS LLM PIPELINE

Main pipeline:
1. Image input
2. Preprocess image
3. Run OCR
4. Clean OCR text
5. Build extraction prompt
6. Extract JSON with LLM
7. Parse JSON
8. Repair small JSON issues
9. Validate fields
10. Run quality checks

Preprocessing:
- Grayscale
- Denoise
- Otsu or adaptive thresholding
- Deskew when text is tilted

OCR cleanup:
- Remove extra spaces
- Remove form feed characters
- Repair common OCR mistakes
- Score text with required keywords

LLM extraction:
- Give OCR text
- Give strict schema
- Ask for valid JSON only
- Use null for missing values

Validation:
- Check required fields
- Check numeric types
- Check date format
- Check conversions <= clicks
- Check spend and total match
'''

print(cheat_sheet)

## Next up: Day 25 — DocumentIntelligenceProject

You will build an end-to-end document extraction mini project for Phase 5.